# Tầng 2 — Rút luật + Giải mã MAP  ·  KHÔNG cần GPU

Hai việc chạy được ngay hôm nay:

| Mục | Việc | Thời gian |
|---|---|---|
| **1** | Tìm augmentation trong `experiment_data.py` + kiểm kê code Tầng 2 có sẵn | 1 phút |
| **2–5** | Rút luật từ nhãn `fit`, lọc help/hurt, giải mã MAP, báo cáo kèm CI | 10–20 phút |

Lõi đã được kiểm thử trên dữ liệu tổng hợp có cài sẵn luật: tìm đúng **2/84** ứng viên là luật thật, loại đúng 82 cái còn lại; R1 thắng C1 +0.57 điểm, R0 thắng C0 +1.61.

## 0. Thiết lập

In [ ]:
import os, sys, json, re, itertools
from pathlib import Path
import numpy as np

try:
    from google.colab import drive
    if not Path('/content/drive').exists(): drive.mount('/content/drive')
except ImportError: pass

SYM    = Path('/content/drive/MyDrive/Sym')
REPO   = SYM/'Contrastive_V1'
RUNS   = SYM/'runs_retry1'
DATA   = SYM/'DISFA'
SPLITS = SYM/'splits'
VARIANTS, FOLDS, SEEDS = ['baseline','current','full'], [1,2,3], [0]
OUT = SYM/'stage2_out'; OUT.mkdir(exist_ok=True)

for n,p in [('REPO',REPO),('RUNS',RUNS),('DATA/list',DATA/'list'),('SPLITS',SPLITS)]:
    print(f'{"OK   " if p.is_dir() else "THIẾU"}  {n:12s} {p}')

## 1. Augmentation hiện có + code Tầng 2 có sẵn

In [ ]:
SRC = RUNS/'baseline'/'fold2'/'seed0'/'source'
ed  = (SRC/'experiment_data.py').read_text(encoding='utf-8')
tc  = (SRC/'train_contrastive.py').read_text(encoding='utf-8')

print('='*72); print('1A. experiment_data.py — transform / augmentation'); print('='*72)
for kw in ['Compose','Resize','CenterCrop','RandomCrop','RandomResizedCrop','Flip','Jitter',
           'Grayscale','Blur','Erasing','ToTensor','Normalize','transform','Dataset','DataLoader']:
    hits=[l.strip() for l in ed.splitlines() if kw in l]
    print(f'  {kw:20s} {len(hits):>2}' + ('   <-- KHÔNG CÓ' if not hits else ''))
    for l in hits[:3]: print(f'        {l[:110]}')

print('\n' + '='*72); print('1B. hàm/lớp dựng dữ liệu trong experiment_data.py'); print('='*72)
for m in re.finditer(r'((?:def|class)\s+\w*(?:[Tt]ransform|[Dd]ataset|[Ll]oader|augment)\w*.*?)(?=\n(?:def|class)\s|\Z)', ed, re.S):
    print(m.group(1)[:1400]); print('-'*60)

print('='*72); print('1C. nơi train_contrastive.py dựng DataLoader'); print('='*72)
for m in re.finditer(r'(def\s+\w+\(.*?)(?=\ndef |\nclass |\Z)', tc, re.S):
    if 'DataLoader' in m.group(1): print(m.group(1)[:1600]); print('-'*60)

print('='*72); print('1D. thư mục con trong Contrastive_V1 (tìm code Tầng 2 có sẵn)'); print('='*72)
for e in sorted(os.scandir(REPO), key=lambda e:e.name):
    if e.is_dir() and not e.name.startswith('.') and e.name != '__pycache__':
        fs = [x.name for x in os.scandir(e.path)][:12]
        print(f'  {e.name}/  {fs}')

## 2. Lõi — rút luật và giải mã

Chạy ô này một lần, nó chỉ định nghĩa hàm.

In [ ]:
"""Lõi Tầng 2: rút luật (bootstrap theo subject) + giải mã MAP 256 cấu hình."""
import numpy as np, itertools
AU_NAMES = ['AU1','AU2','AU4','AU6','AU9','AU12','AU25','AU26']
NA = 8
Y256 = np.array(list(itertools.product([0,1], repeat=NA)), dtype=np.int8)   # (256, 8)

# ---------- thống kê đủ theo subject ----------
def subject_stats(y, sub):
    """y:(N,8) 0/1, sub:(N,) -> cnt:(S,), m:(S,8), mm:(S,8,8)"""
    subs = np.unique(sub); S = len(subs)
    cnt = np.zeros(S); m = np.zeros((S, NA)); mm = np.zeros((S, NA, NA))
    for k, s in enumerate(subs):
        ys = y[sub == s].astype(float)
        cnt[k] = len(ys); m[k] = ys.sum(0); mm[k] = ys.T @ ys
    return subs, cnt, m, mm

def _boot_idx(S, B, rng):
    return rng.integers(0, S, size=(B, S))

# ---------- rút luật ----------
def induce_rules(y_fit, sub_fit, B=1000, seed=0,
                 ucb_max=0.8, min_E=50, min_frac_sub=0.70, min_conf_lcb=0.60, w_cap=5.0):
    rng = np.random.default_rng(seed)
    subs, cnt, m, mm = subject_stats(y_fit, sub_fit)
    S = len(subs); bi = _boot_idx(S, B, rng)
    Cb = cnt[bi].sum(1)                       # (B,)
    Mb = m[bi].sum(1)                         # (B, 8)
    MMb = mm[bi].sum(1)                       # (B, 8, 8)
    n0, a0, aa0 = cnt.sum(), m.sum(0), mm.sum(0)
    rules = []

    def finish(kind, i, j, lv_b, lv0, E, frac, extra=None):
        ucb = float(np.percentile(lv_b, 97.5))
        ok  = (ucb < ucb_max) and (E >= min_E) and (frac >= min_frac_sub)
        if extra is not None: ok = ok and extra['ok']
        w0 = float(min(max(-np.log(max(ucb, 1e-12)), 0.0), w_cap))
        r = dict(kind=kind, i=i, j=j,
                 name=(f'¬{AU_NAMES[i]} ∨ ¬{AU_NAMES[j]}' if kind=='excl'
                       else f'{AU_NAMES[i]} → {AU_NAMES[j]}'),
                 lift=float(lv0), ucb95=ucb, E=float(E), frac_sub=float(frac),
                 w0=w0, accepted=bool(ok))
        if extra is not None: r.update({k:v for k,v in extra.items() if k!='ok'})
        rules.append(r)

    for i, j in itertools.combinations(range(NA), 2):          # LOẠI TRỪ
        num = MMb[:, i, j] * Cb; den = Mb[:, i] * Mb[:, j]
        lv_b = np.where(den > 0, num / np.maximum(den, 1e-9), 1.0)
        lv0  = (aa0[i, j] * n0) / max(a0[i] * a0[j], 1e-9)
        E    = a0[i] * a0[j] / max(n0, 1)
        ps   = np.where((m[:, i] > 0) & (m[:, j] > 0),
                        (mm[:, i, j] * cnt) / np.maximum(m[:, i] * m[:, j], 1e-9), np.nan)
        frac = np.nanmean(ps < 1.0) if np.isfinite(ps).any() else 0.0
        finish('excl', i, j, lv_b, lv0, E, frac)

    for i, j in itertools.permutations(range(NA), 2):          # KÉO THEO i -> j
        vb  = Mb[:, i] - MMb[:, i, j]
        den = Mb[:, i] * (Cb - Mb[:, j])
        lv_b = np.where(den > 0, (vb * Cb) / np.maximum(den, 1e-9), 1.0)
        v0   = a0[i] - aa0[i, j]
        lv0  = (v0 * n0) / max(a0[i] * (n0 - a0[j]), 1e-9)
        E    = a0[i] * (n0 - a0[j]) / max(n0, 1)
        cb   = np.where(Mb[:, i] > 0, MMb[:, i, j] / np.maximum(Mb[:, i], 1e-9), 0.0)
        conf_lcb = float(np.percentile(cb, 2.5))
        ps   = np.where(m[:, i] > 0,
                        ((m[:, i]-mm[:, i, j]) * cnt) / np.maximum(m[:, i]*(cnt-m[:, j]), 1e-9), np.nan)
        frac = np.nanmean(ps < 1.0) if np.isfinite(ps).any() else 0.0
        finish('impl', i, j, lv_b, lv0, E, frac,
               extra=dict(conf=float(aa0[i, j]/max(a0[i],1e-9)), conf_lcb=conf_lcb,
                          ok=(conf_lcb >= min_conf_lcb)))
    return rules

# ---------- ma trận vi phạm ----------
def violation_matrix(rules):
    V = np.zeros((256, len(rules)), dtype=np.float32)
    for r_, r in enumerate(rules):
        i, j = r['i'], r['j']
        V[:, r_] = ((Y256[:, i] == 1) & (Y256[:, j] == 1)) if r['kind'] == 'excl' \
                   else ((Y256[:, i] == 1) & (Y256[:, j] == 0))
    return V

def help_hurt(rules, y, p, th=0.5):
    yh = p > th; out = []
    for r in rules:
        i, j = r['i'], r['j']
        if r['kind'] == 'excl':
            viol = yh[:, i] & yh[:, j]; tv = y[:, i].astype(bool) & y[:, j].astype(bool)
        else:
            viol = yh[:, i] & ~yh[:, j]; tv = y[:, i].astype(bool) & ~y[:, j].astype(bool)
        out.append((int((viol & ~tv).sum()), int((viol & tv).sum())))
    return out

# ---------- F1 ----------
def macro_f1(y, pred):
    f = []
    for i in range(NA):
        tp = float((pred[:, i] & (y[:, i] > 0.5)).sum())
        fp = float((pred[:, i] & ~(y[:, i] > 0.5)).sum())
        fn = float((~pred[:, i] & (y[:, i] > 0.5)).sum())
        f.append(2*tp / max(2*tp + fp + fn, 1e-9))
    return float(np.mean(f)), f

# ---------- giải mã ----------
def tune_thresholds(y, p, grid=np.arange(0.05, 0.96, 0.05)):
    th = np.full(NA, 0.5)
    for i in range(NA):
        best, bt = -1, 0.5
        for t in grid:
            pr = p[:, i] > t
            tp = float((pr & (y[:, i] > 0.5)).sum()); fp = float((pr & ~(y[:, i] > 0.5)).sum())
            fn = float((~pr & (y[:, i] > 0.5)).sum())
            f1 = 2*tp / max(2*tp + fp + fn, 1e-9)
            if f1 > best: best, bt = f1, float(t)
        th[i] = bt
    return th

def map_decode(p, V, w, lam, bias=None, chunk=20000):
    """argmax_y  Σ_i y_i (logit_i − b_i)  −  λ Σ_r w_r V[y,r]"""
    L = np.log(np.clip(p, 1e-7, 1-1e-7)) - np.log(np.clip(1-p, 1e-7, 1-1e-7))
    if bias is not None: L = L - bias[None, :]
    pen = (V @ w).astype(np.float64) * lam              # (256,)
    Yf  = Y256.astype(np.float64)
    out = np.zeros((len(p), NA), dtype=bool)
    for s in range(0, len(p), chunk):
        sc = L[s:s+chunk] @ Yf.T - pen[None, :]
        out[s:s+chunk] = Y256[np.argmax(sc, axis=1)].astype(bool)
    return out

def logit(t): return np.log(np.clip(t,1e-6,1-1e-6)) - np.log(np.clip(1-t,1e-6,1-1e-6))

## 3. Rút luật từ nhãn `fit`

Bootstrap 1000 lần **theo subject**. Cả hai loại luật dùng chung một thước đo: lift của **mẫu vi phạm** — loại trừ → `(i=1, j=1)`, kéo theo → `(i=1, j=0)`.

Nhận khi `UCB95(lift_v) < 0.8`, `E ≥ 50`, ≥70% subject cùng chiều; kéo theo thêm `LCB95(P(j|i)) ≥ 0.6`. Trọng số `w0 = −log UCB95`, chặn trên 5.

Bảng luật **chỉ từ nhãn `fit`** nên dùng chung cho mọi variant và seed.

In [ ]:
def load_fit(fold):
    paths  = (DATA/'list'/f'DISFA_train_img_path_fold{fold}.txt').read_text(encoding='utf-8').split()
    labels = np.loadtxt(DATA/'list'/f'DISFA_train_label_fold{fold}.txt')
    mf  = json.loads((SPLITS/f'fold{fold}.json').read_text(encoding='utf-8'))
    idx = np.array(mf['fit_indices'])
    y   = (labels[idx] > 0.5).astype(int)
    sub = np.array([p.split('/')[0] for p in paths])[idx]
    return y, sub, mf

RULES = {}
for f in FOLDS:
    y_fit, sub_fit, mf = load_fit(f)
    print(f'\nfold{f}: fit {len(y_fit):,} frame · {len(np.unique(sub_fit))} subject '
          f'{sorted(set(map(str, sub_fit)))[:4]}...')
    rs  = induce_rules(y_fit, sub_fit, B=1000, seed=2026)
    acc = [r for r in rs if r['accepted']]
    RULES[f] = dict(all=rs, acc=acc)
    print(f'  nhận {len(acc)}/{len(rs)} ứng viên')
    print(f'  {"luật":<24}{"loại":<6}{"lift":>8}{"UCB95":>8}{"E":>9}{"%sub":>7}{"w0":>7}')
    print('  ' + '-'*70)
    for r in sorted(acc, key=lambda r:-r['w0']):
        print(f'  {r["name"]:<24}{r["kind"]:<6}{r["lift"]:>8.3f}{r["ucb95"]:>8.3f}'
              f'{r["E"]:>9.0f}{r["frac_sub"]:>7.2f}{r["w0"]:>7.2f}'
              + (f'  conf={r["conf"]:.2f}(LCB {r["conf_lcb"]:.2f})' if r['kind']=='impl' else ''))

## 4. Lọc `help` − `hurt` trên inner val

Bỏ luật có `help − hurt ≤ 0`. Dùng dự đoán của `baseline` để bảng luật không phụ thuộc variant.

In [ ]:
def npz_of(v,f,s,split):
    d = RUNS/v/f'fold{f}'/f'seed{s}'/('evaluation' if split=='test' else 'evaluation_val')
    hit = sorted(d.glob(f'{split}_predictions.npz')) or sorted(d.glob(f'*{split}*.npz'))
    if not hit: return None
    z = np.load(hit[0], allow_pickle=True); return {k:z[k] for k in z.files}

def subj(d):
    if 'subjects' in d: return np.asarray(d['subjects']).astype(str)
    return np.array([str(p).split('/')[0] for p in d['paths']])

for f in FOLDS:
    dv  = npz_of('baseline', f, 0, 'val')
    if dv is None: print(f'fold{f}: THIẾU val npz'); continue
    y, p = (dv['labels']>0.5).astype(int), dv['probabilities']
    acc = RULES[f]['acc']
    hh  = help_hurt(acc, y, p)
    keep = []
    print(f'\nfold{f}:  {"luật":<24}{"help":>8}{"hurt":>8}{"net":>8}   giữ?')
    print('  ' + '-'*58)
    for r,(h,k) in zip(acc, hh):
        r['help'], r['hurt'] = h, k
        ok = (h - k) > 0
        if ok: keep.append(r)
        print(f'  {r["name"]:<24}{h:>8}{k:>8}{h-k:>8}   {"giữ" if ok else "BỎ"}')
    RULES[f]['keep'] = keep
    print(f'  -> còn {len(keep)}/{len(acc)} luật')

import csv
for f in FOLDS:
    with open(OUT/f'rules_fold{f}.csv','w',newline='',encoding='utf-8') as fh:
        wtr = csv.DictWriter(fh, fieldnames=['name','kind','i','j','lift','ucb95','E',
                                             'frac_sub','w0','accepted','help','hurt','conf','conf_lcb'])
        wtr.writeheader()
        for r in RULES[f]['all']: wtr.writerow({k:r.get(k,'') for k in wtr.fieldnames})
print(f'\nĐã lưu rules_fold*.csv vào {OUT}')

## 5. Giải mã MAP + báo cáo

Bốn nhánh: **C0** ngưỡng 0.5 · **C1** ngưỡng riêng từng AU chọn trên val · **R0** MAP có luật bias 0 · **R1** MAP có luật bias = ngưỡng C1. `λ` chọn trên val, **có λ=0 trong lưới** nên luật tự tắt nếu vô ích. `R*` = nhánh tốt hơn giữa R0/R1 theo val.

In [ ]:
LAM = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
NB  = 500
rows = []

def boot_delta(y, pa, pb, sub, B=NB, seed=7):
    rng = np.random.default_rng(seed); S = np.unique(sub)
    idx = {s: np.where(sub==s)[0] for s in S}
    d = []
    for _ in range(B):
        take = np.concatenate([idx[s] for s in rng.choice(S, len(S), replace=True)])
        d.append(macro_f1(y[take], pa[take])[0] - macro_f1(y[take], pb[take])[0])
    return float(np.percentile(d,2.5))*100, float(np.percentile(d,97.5))*100

for v in VARIANTS:
    for f in FOLDS:
        dv, dt = npz_of(v,f,0,'val'), npz_of(v,f,0,'test')
        if dv is None or dt is None: print(f'{v}/fold{f}: thiếu npz'); continue
        yv, pv = (dv['labels']>0.5).astype(int), dv['probabilities']
        yt, pt = (dt['labels']>0.5).astype(int), dt['probabilities']
        st = subj(dt)
        keep = RULES[f]['keep']
        V = violation_matrix(keep); w = np.array([r['w0'] for r in keep]) if keep else np.zeros(0)

        pred = {}
        pred['C0'] = pt > 0.5
        th = tune_thresholds(yv, pv); pred['C1'] = pt > th[None,:]
        best = {}
        for nm, bias in [('R0', None), ('R1', logit(th))]:
            if len(keep)==0: pred[nm], best[nm] = pred['C0' if nm=='R0' else 'C1'], 0.0; continue
            bf, bl = -1, 0.0
            for lam in LAM:
                s_ = macro_f1(yv, map_decode(pv,V,w,lam,bias))[0]
                if s_ > bf: bf, bl = s_, lam
            best[nm] = bl; pred[nm] = map_decode(pt,V,w,bl,bias)
        vR0 = macro_f1(yv, map_decode(pv,V,w,best['R0'],None))[0] if len(keep) else 0
        vR1 = macro_f1(yv, map_decode(pv,V,w,best['R1'],logit(th)))[0] if len(keep) else 0
        star = 'R1' if vR1 >= vR0 else 'R0'
        rows.append(dict(variant=v, fold=f, star=star, lam=best[star],
                         **{k: 100*macro_f1(yt, pred[k])[0] for k in ['C0','C1','R0','R1']},
                         ci0=boot_delta(yt, pred[star], pred['C0'], st),
                         ci1=boot_delta(yt, pred[star], pred['C1'], st)))
        r = rows[-1]
        print(f'{v}/fold{f}: C0={r["C0"]:.2f} C1={r["C1"]:.2f} R0={r["R0"]:.2f} R1={r["R1"]:.2f}'
              f'  R*={star}(λ={r["lam"]})')

print('\n' + '='*86); print('TỔNG HỢP — trung bình 3 fold (seed 0)'); print('='*86)
print(f'{"variant":<10}{"C0":>8}{"C1":>8}{"R0":>8}{"R1":>8}{"R*":>8}{"Δ R*-C0":>10}{"Δ R*-C1":>10}')
print('-'*86)
for v in VARIANTS:
    rr = [r for r in rows if r['variant']==v]
    if not rr: continue
    m = {k: np.mean([r[k] for r in rr]) for k in ['C0','C1','R0','R1']}
    ms = np.mean([r[r['star']] for r in rr])
    print(f'{v:<10}{m["C0"]:>8.2f}{m["C1"]:>8.2f}{m["R0"]:>8.2f}{m["R1"]:>8.2f}{ms:>8.2f}'
          f'{ms-m["C0"]:>+10.2f}{ms-m["C1"]:>+10.2f}')

print('\nCI 95% bootstrap theo subject, từng fold:')
for r in rows:
    print(f'  {r["variant"]}/fold{r["fold"]}  R*-C0 [{r["ci0"][0]:+.2f}, {r["ci0"][1]:+.2f}]'
          f'   R*-C1 [{r["ci1"][0]:+.2f}, {r["ci1"][1]:+.2f}]')

json.dump([{k:(list(v_) if isinstance(v_,tuple) else v_) for k,v_ in r.items()} for r in rows],
          open(OUT/'stage2_results.json','w'), indent=1)
print(f'\nĐã lưu {OUT}/stage2_results.json')
print('\nTIÊU CHÍ GO/NO-GO: R* phải thắng CẢ C0 LẪN C1, và CI không chứa 0.')

## Gửi lại gì

1. **Mục 1** — augmentation và `DataLoader`. Mảnh ghép cuối để tôi viết lệnh train mới.
2. **Mục 3** — bảng luật mỗi fold. Xem `AU4–AU12` và `AU9–AU12` có qua được kiểm định không.
3. **Mục 5** — bảng tổng hợp và CI. Đây là kết quả Tầng 2 chính thức.